# Substitution analysis

Analyze substituted training candidates for one model and one generated-sample subset.

In [ ]:
from __future__ import annotations

import gzip
import os
import pickle
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import __main__

os.environ.setdefault(
    "MPLCONFIGDIR", f"/scratch_local/matplotlib-{os.environ.get('USER', 'codex')}"
)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or shell working directory."""
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src").is_dir():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
SCRIPTS_DIR = ROOT / "scripts"
for import_path in (ROOT, NOTEBOOKS_DIR, SCRIPTS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from plot_style import GRAY, PALETTE, apply_plot_style  # noqa: E402
from substitute_structures import SubstitutedEntry  # noqa: E402

from src._subst_cost import subst_cost_mod_petti  # noqa: E402
from src.config import INPUT_DIR, RESULTS_DIR  # noqa: E402

# Some existing substituted-entry pickles were written with SubstitutedEntry
# resolved from __main__. Provide that symbol before unpickling.
__main__.SubstitutedEntry = SubstitutedEntry

ROOT, INPUT_DIR, RESULTS_DIR

In [ ]:
MODEL = "mattergen"
SUBSET = "metastable_smact_valid"

SUBSET_OPTIONS = {"all", "metastable", "metastable_smact_valid"}
METASTABLE_EHULL_MAX = 0.1

GEN_FILE = "relaxed_niggli.pkl.gz"
TRAIN_FILE = "train.pkl.gz"
EHULL_RELAXED_FILE = "ehull_relaxed.pkl.gz"
SMACT_VALIDITY_FILE = "smact_validity.npz"

SM_ANON_MATCH_FILE = "sm_anon.pkl.gz"
WYCKOFF_MATCH_FILE = "wyckoff_match_s=0.01_c=mod_petti.pkl.gz"

TOP3_SM_ANON_FILE = "substituted_top3_sm_anon.pkl.gz"
TOP3_WYCKOFF_FILE = "substituted_top3_wyckoff_match_s=0.01_c=mod_petti.pkl.gz"
RELAXED_SM_ANON_FILE = "substituted_relaxed_niggli_top3_sm_anon_gen_matches.pkl.gz"
RELAXED_WYCKOFF_FILE = (
    "substituted_relaxed_niggli_top3_wyckoff_match_s=0.01_c="
    "mod_petti_gen_matches.pkl.gz"
)

SOURCE_ORDER = ["anon", "wyckoff"]
MATCH_SETTING_ORDER = ["match_anon", "match_wyckoff", "match_concat"]
MATCH_COLORS = {"relaxed match": PALETTE[0], "no relaxed match": GRAY}

if SUBSET not in SUBSET_OPTIONS:
    raise ValueError(f"SUBSET must be one of {sorted(SUBSET_OPTIONS)}, got {SUBSET!r}")


def load_pickle_gz(path: Path) -> Any:
    with gzip.open(path, "rb") as f:
        return pickle.load(f)  # noqa: S301


def required_paths(model: str) -> dict[str, Path]:
    gen_dir = INPUT_DIR / "gen" / "preprocessed" / model
    result_dir = RESULTS_DIR / model
    train_dir = INPUT_DIR / "train" / "preprocessed"
    return {
        "generated_structures": gen_dir / GEN_FILE,
        "training_structures": train_dir / TRAIN_FILE,
        "ehull_relaxed": gen_dir / EHULL_RELAXED_FILE,
        "smact_validity": gen_dir / SMACT_VALIDITY_FILE,
        "sm_anon_matches": result_dir / SM_ANON_MATCH_FILE,
        "wyckoff_matches": result_dir / WYCKOFF_MATCH_FILE,
        "top3_sm_anon": result_dir / TOP3_SM_ANON_FILE,
        "top3_wyckoff": result_dir / TOP3_WYCKOFF_FILE,
        "relaxed_sm_anon": result_dir / RELAXED_SM_ANON_FILE,
        "relaxed_wyckoff": result_dir / RELAXED_WYCKOFF_FILE,
    }


def check_required_paths(paths: dict[str, Path]) -> None:
    missing = {kind: path for kind, path in paths.items() if not path.exists()}
    if missing:
        missing_lines = "\n".join(f"- {kind}: {path}" for kind, path in missing.items())
        raise FileNotFoundError(f"Missing required input files:\n{missing_lines}")


paths = required_paths(MODEL)
check_required_paths(paths)
paths

In [ ]:
def validate_1d_length(values: np.ndarray, expected: int, path: Path) -> np.ndarray:
    if values.ndim != 1 or len(values) != expected:
        raise ValueError(f"{path} has shape {values.shape}; expected ({expected},)")
    return values


def load_smact_validity(path: Path, expected: int) -> np.ndarray:
    with np.load(path) as data:
        if "valid" not in data.files:
            raise ValueError(f"{path} does not contain a 'valid' array")
        values = np.asarray(data["valid"], dtype=bool)
    return validate_1d_length(values, expected, path)


def build_subset_mask(model: str, subset: str) -> pd.Series:
    gen_count = len(load_pickle_gz(paths["generated_structures"]))
    ehull_relaxed = validate_1d_length(
        np.asarray(load_pickle_gz(paths["ehull_relaxed"]), dtype=float),
        gen_count,
        paths["ehull_relaxed"],
    )
    smact_valid = load_smact_validity(paths["smact_validity"], gen_count)

    if subset == "all":
        mask = np.ones(gen_count, dtype=bool)
    elif subset == "metastable":
        mask = ehull_relaxed <= METASTABLE_EHULL_MAX
    elif subset == "metastable_smact_valid":
        mask = (ehull_relaxed <= METASTABLE_EHULL_MAX) & smact_valid
    else:
        raise ValueError(f"Unknown subset: {subset!r}")

    return pd.Series(mask, index=pd.RangeIndex(gen_count, name="gen_idx"))


subset_mask = build_subset_mask(MODEL, SUBSET)
selected_gen_indices = set(subset_mask[subset_mask].index.astype(int))

display(
    Markdown(
        f"**Model:** `{MODEL}`  \n"
        f"**Subset:** `{SUBSET}`  \n"
        f"**Generated samples in subset:** {len(selected_gen_indices):,}"
    )
)

In [ ]:
def candidate_key(row: pd.Series | dict[str, Any]) -> tuple[int, int, float, float]:
    return (
        int(row["gen_idx"]),
        int(row["train_idx"]),
        round(float(row["cost_uniform"]), 12),
        round(float(row["cost_mod_petti"]), 12),
    )


def entries_to_frame(
    entries: list[SubstitutedEntry],
    records: list[dict[str, Any]],
    source: str,
) -> pd.DataFrame:
    if len(entries) != len(records):
        raise ValueError(
            f"{source}: entries length {len(entries)} != records length {len(records)}"
        )

    rows: list[dict[str, Any]] = []
    for entry_idx, (entry, record) in enumerate(zip(entries, records, strict=True)):
        if int(record["entry_idx"]) != entry_idx:
            raise ValueError(
                f"{source}: record entry_idx={record['entry_idx']} "
                f"at position {entry_idx}"
            )
        for attr in ("gen_idx", "train_idx", "rank"):
            if int(getattr(entry, attr)) != int(record[attr]):
                raise ValueError(f"{source}: {attr} mismatch at entry {entry_idx}")
        for attr in ("cost_uniform", "cost_mod_petti"):
            if not np.isclose(float(getattr(entry, attr)), float(record[attr])):
                raise ValueError(f"{source}: {attr} mismatch at entry {entry_idx}")

        rows.append(
            {
                "source": source,
                "source_order": SOURCE_ORDER.index(source),
                "entry_idx": entry_idx,
                "gen_idx": int(entry.gen_idx),
                "train_idx": int(entry.train_idx),
                "rank": int(entry.rank),
                "cost_uniform": float(entry.cost_uniform),
                "cost_mod_petti": float(entry.cost_mod_petti),
                "match": bool(record["match"]),
            }
        )

    frame = pd.DataFrame(rows)
    frame["match_label"] = np.where(frame["match"], "relaxed match", "no relaxed match")
    return frame


def load_candidates() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    anon = entries_to_frame(
        load_pickle_gz(paths["top3_sm_anon"]),
        load_pickle_gz(paths["relaxed_sm_anon"]),
        "anon",
    )
    wyckoff = entries_to_frame(
        load_pickle_gz(paths["top3_wyckoff"]),
        load_pickle_gz(paths["relaxed_wyckoff"]),
        "wyckoff",
    )

    anon = anon[anon["gen_idx"].isin(selected_gen_indices)].reset_index(drop=True)
    wyckoff = wyckoff[wyckoff["gen_idx"].isin(selected_gen_indices)].reset_index(
        drop=True
    )
    concat = pd.concat([anon, wyckoff], ignore_index=True)
    return anon, wyckoff, concat


def summarize_candidates(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for setting, table in tables.items():
        rows.append(
            {
                "setting": setting,
                "candidates": len(table),
                "generated_samples": table["gen_idx"].nunique(),
                "relaxed_match_candidates": int(table["match"].sum()),
            }
        )
    return pd.DataFrame(rows)


match_anon, match_wyckoff, match_concat = load_candidates()
candidate_tables = {
    "match_anon": match_anon,
    "match_wyckoff": match_wyckoff,
    "match_concat": match_concat,
}

summarize_candidates(candidate_tables)

In [ ]:
def select_representatives(candidates: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty:
        return candidates.copy()

    pool = candidates.copy()
    pool["has_any_relaxed_match"] = pool.groupby("gen_idx")["match"].transform("any")
    pool = pool[~pool["has_any_relaxed_match"] | pool["match"]].copy()
    pool = pool.sort_values(
        [
            "gen_idx",
            "cost_mod_petti",
            "cost_uniform",
            "source_order",
            "rank",
            "train_idx",
            "entry_idx",
        ],
        kind="mergesort",
    )
    return pool.drop_duplicates("gen_idx", keep="first").reset_index(drop=True)


representative_tables = {
    setting: select_representatives(table)
    for setting, table in candidate_tables.items()
}

representative_summary = pd.DataFrame(
    {
        "setting": setting,
        "representatives": len(table),
        "relaxed_match_representatives": int(table["match"].sum())
        if not table.empty
        else 0,
        "unmatched_representatives": int((~table["match"]).sum())
        if not table.empty
        else 0,
    }
    for setting, table in representative_tables.items()
)
representative_summary

## Candidate tables

In [ ]:
for setting in MATCH_SETTING_ORDER:
    display(Markdown(f"### `{setting}`"))
    display(candidate_tables[setting])

## Representative tables

In [ ]:
for setting in MATCH_SETTING_ORDER:
    display(Markdown(f"### `{setting}`"))
    display(representative_tables[setting])

## Cost distributions

In [ ]:
apply_plot_style()


def plot_cost_distributions(representatives: dict[str, pd.DataFrame]) -> None:
    for setting in MATCH_SETTING_ORDER:
        table = representatives[setting]
        if table.empty:
            display(Markdown(f"### `{setting}`\n\nNo representatives available."))
            continue

        fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
        for ax, column, xlabel in zip(
            axes,
            ["cost_mod_petti", "cost_uniform"],
            ["mod-Pettifor cost", "uniform cost"],
            strict=True,
        ):
            sns.histplot(
                data=table,
                x=column,
                hue="match_label",
                hue_order=["relaxed match", "no relaxed match"],
                palette=MATCH_COLORS,
                multiple="layer",
                element="step",
                stat="count",
                common_norm=False,
                ax=ax,
            )
            ax.set_xlabel(xlabel)
            ax.set_ylabel("Representative count")
        fig.suptitle(f"{setting}: {MODEL}, {SUBSET}")
        plt.show()


plot_cost_distributions(representative_tables)

## Top substitutions

In [ ]:
def load_match_lookup(
    path: Path,
    selected: pd.DataFrame,
) -> tuple[dict[tuple[int, int, float, float], Any], pd.DataFrame]:
    needed_keys = {candidate_key(row) for _, row in selected.iterrows()}
    lookup: dict[tuple[int, int, float, float], Any] = {}
    duplicate_counts: Counter[tuple[int, int, float, float]] = Counter()

    if not needed_keys:
        return lookup, pd.DataFrame()

    for match in load_pickle_gz(path):
        key = (
            int(match.idx1),
            int(match.idx2),
            round(float(match.cost_uniform), 12),
            round(float(match.cost_mod_petti), 12),
        )
        if key not in needed_keys:
            continue
        if key in lookup:
            duplicate_counts[key] += 1
        else:
            lookup[key] = match

    warning_rows = []
    for key in sorted(needed_keys):
        if key not in lookup:
            warning_rows.append(
                {
                    "gen_idx": key[0],
                    "train_idx": key[1],
                    "cost_uniform": key[2],
                    "cost_mod_petti": key[3],
                    "issue": "missing original match object",
                    "extra_duplicates": 0,
                }
            )
        elif duplicate_counts[key]:
            warning_rows.append(
                {
                    "gen_idx": key[0],
                    "train_idx": key[1],
                    "cost_uniform": key[2],
                    "cost_mod_petti": key[3],
                    "issue": "duplicate original match key; first used",
                    "extra_duplicates": int(duplicate_counts[key]),
                }
            )

    return lookup, pd.DataFrame(warning_rows)


def substitution_pairs_anon(
    match: Any, gen_structures: list[Any], train_structures: list[Any]
) -> list[tuple[str, str]]:
    gen_species = [site.specie.symbol for site in gen_structures[match.idx1]]
    train_species = [site.specie.symbol for site in train_structures[match.idx2]]

    pairs = []
    if match.s1_supercell:
        # mapping[i] = j: train atom i -> generated supercell atom j.
        for train_atom_idx, gen_supercell_atom_idx in enumerate(match.mapping):
            pairs.append(
                (
                    train_species[train_atom_idx],
                    gen_species[int(gen_supercell_atom_idx) // match.fu],
                )
            )
    else:
        # mapping[i] = j: train supercell atom i -> generated atom j.
        for train_supercell_atom_idx, gen_atom_idx in enumerate(match.mapping):
            pairs.append(
                (
                    train_species[train_supercell_atom_idx // match.fu],
                    gen_species[int(gen_atom_idx)],
                )
            )
    return pairs


def substitution_pairs_wyckoff(
    match: Any, gen_structures: list[Any], train_structures: list[Any]
) -> list[tuple[str, str]]:
    gen_species = [site.specie.symbol for site in gen_structures[match.idx1]]
    train_species = [site.specie.symbol for site in train_structures[match.idx2]]
    return [
        (train_species[train_atom_idx], gen_species[int(gen_atom_idx)])
        for train_atom_idx, gen_atom_idx in enumerate(match.atom_map)
    ]


def build_top_substitutions(
    representatives: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    matched_by_source = {
        source: pd.concat(
            [
                table[(table["match"]) & (table["source"] == source)]
                for table in representatives.values()
            ],
            ignore_index=True,
        ).drop_duplicates(["gen_idx", "train_idx", "cost_uniform", "cost_mod_petti"])
        for source in SOURCE_ORDER
    }

    anon_lookup, anon_warnings = load_match_lookup(
        paths["sm_anon_matches"], matched_by_source["anon"]
    )
    wyckoff_lookup, wyckoff_warnings = load_match_lookup(
        paths["wyckoff_matches"], matched_by_source["wyckoff"]
    )

    gen_structures = load_pickle_gz(paths["generated_structures"])
    train_structures = load_pickle_gz(paths["training_structures"])

    lookups = {"anon": anon_lookup, "wyckoff": wyckoff_lookup}
    pair_builders = {
        "anon": substitution_pairs_anon,
        "wyckoff": substitution_pairs_wyckoff,
    }

    rows = []
    for setting, table in representatives.items():
        counts: Counter[tuple[str, str]] = Counter()
        for _, row in table[table["match"]].iterrows():
            source = str(row["source"])
            match = lookups[source].get(candidate_key(row))
            if match is None:
                continue
            for train_element, gen_element in pair_builders[source](
                match, gen_structures, train_structures
            ):
                if train_element != gen_element:
                    counts[(train_element, gen_element)] += 1

        total = sum(counts.values())
        for (train_element, gen_element), count in counts.most_common(10):
            rows.append(
                {
                    "setting": setting,
                    "substitution": f"{train_element} -> {gen_element}",
                    "train_element": train_element,
                    "gen_element": gen_element,
                    "cost_mod_petti": subst_cost_mod_petti(train_element, gen_element),
                    "count": count,
                    "percent": 100 * count / total if total else 0.0,
                }
            )

    warnings = pd.concat(
        [
            anon_warnings.assign(source="anon"),
            wyckoff_warnings.assign(source="wyckoff"),
        ],
        ignore_index=True,
    )
    return pd.DataFrame(rows), warnings


top_substitutions, substitution_warnings = build_top_substitutions(
    representative_tables
)

if not substitution_warnings.empty:
    display(Markdown("### Match-object lookup warnings"))
    display(substitution_warnings)

top_substitutions